# Prepare Offline Assets

**Chay tren Kaggle: Internet BAT, KHONG can GPU.**

Download:
- segmentation-models-pytorch + dependencies
- 13 pretrained encoder weights (15 experiments)

### Sau khi chay xong:
1. **Save Version** (Quick Save)
2. Mo notebook train -> **Add Data** -> tab **Notebook Output** -> chon notebook nay -> **Add**

In [ ]:
import subprocess, sys, os, shutil
from pathlib import Path

OUT = Path('/kaggle/working/offline_assets')
PKG_DIR = OUT / 'packages'
PKG_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', OUT)

## 1. Download Packages

In [ ]:
# Download smp + tat ca dependencies
PACKAGES = [
    'segmentation-models-pytorch',
    'albumentations',
    'fvcore',     # tinh FLOPs
    'seaborn',    # visualization
]

for pkg in PACKAGES:
    print(f'\n=== {pkg} ===')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'download',
        pkg, '-d', str(PKG_DIR),
    ])

pkg_files = sorted(PKG_DIR.iterdir())
total_mb = sum(f.stat().st_size for f in pkg_files) / 1024**2
print(f'\nDownloaded {len(pkg_files)} packages ({total_mb:.0f} MB)')

## 2. Download ALL Pretrained Encoder Weights

13 encoders cho 15 experiments (4 groups A-D).

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'segmentation-models-pytorch', 'timm', '-q'])

import segmentation_models_pytorch as smp
import torch

# === 15 experiments, 13 unique encoders ===
ENCODERS = [
    # Group A: CNN Baselines
    ('resnet34',   'A1: UNet + ResNet34'),
    ('resnet50',   'A4: UNet++ + ResNet50'),
    ('resnet101',  'A2/A3: UNet/DeepLabV3+ + ResNet101'),

    # Group B: Modern CNN
    ('efficientnet-b5',    'B1: UNet + EfficientNet-B5'),
    ('tu-convnext_base',   'B2: UPerNet + ConvNeXt-Base'),
    ('tu-convnext_large',  'B3: UPerNet + ConvNeXt-Large'),

    # Group C: Transformer
    ('mit_b3',  'C1/D1-D4: SegFormer + MiT-B3 (+ ablation)'),
    ('mit_b5',  'C2: SegFormer + MiT-B5'),
    ('tu-swinv2_base_window12to24_192to384',  'C3: UPerNet + SwinV2-Base'),
    ('tu-swinv2_large_window12to24_192to384', 'C4: UPerNet + SwinV2-Large'),
]

print(f'Downloading {len(ENCODERS)} encoders...\n')

results = []
for i, (name, desc) in enumerate(ENCODERS, 1):
    print(f'[{i}/{len(ENCODERS)}] {name}')
    print(f'  {desc}')
    try:
        # Dung Unet lam wrapper de trigger download weights
        m = smp.Unet(encoder_name=name, encoder_weights='imagenet',
                     in_channels=3, classes=11)
        params = sum(p.numel() for p in m.parameters())
        print(f'  [OK] {params:,} params')
        results.append((name, 'OK', params))
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception as e:
        print(f'  [ERROR] {e}')
        results.append((name, f'ERROR: {e}', 0))

print('\n=== Summary ===')
ok = sum(1 for _, s, _ in results if s == 'OK')
print(f'{ok}/{len(ENCODERS)} encoders downloaded successfully')
for name, status, _ in results:
    icon = 'V' if status == 'OK' else 'X'
    print(f'  [{icon}] {name}')

## 3. Copy Cache to Output

Giu nguyen cau truc `~/.cache/` de notebook train co the khoi phuc.

In [ ]:
CACHE_ROOT = Path(os.path.expanduser('~/.cache'))
CACHE_OUT = OUT / 'dot_cache'

CACHE_DIRS = [
    CACHE_ROOT / 'huggingface',
    CACHE_ROOT / 'torch',
]

total_files = 0
total_mb = 0

for cache_dir in CACHE_DIRS:
    if not cache_dir.exists():
        print(f'[SKIP] {cache_dir} not found')
        continue
    print(f'\nCopying {cache_dir}...')
    for src_file in cache_dir.rglob('*'):
        if not src_file.is_file():
            continue
        rel = src_file.relative_to(CACHE_ROOT)
        dst = CACHE_OUT / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            src_real = src_file.resolve()
            shutil.copy2(str(src_real), str(dst))
            sz = dst.stat().st_size / 1024**2
            total_mb += sz
            total_files += 1
            if total_files <= 30:
                print(f'  {rel} ({sz:.1f} MB)')

if total_files > 30:
    print(f'  ... va {total_files - 30} files nua')
print(f'\nCopied {total_files} files ({total_mb:.0f} MB)')

## 4. Verify

In [ ]:
print('=' * 60)
print('VERIFICATION')
print('=' * 60)

# Packages
pkg_files = [f for f in PKG_DIR.rglob('*') if f.is_file()]
pkg_mb = sum(f.stat().st_size for f in pkg_files) / 1024**2
print(f'\npackages/   {len(pkg_files):3d} files  {pkg_mb:7.0f} MB')

# Cache
cache_files = [f for f in CACHE_OUT.rglob('*') if f.is_file()]
cache_mb = sum(f.stat().st_size for f in cache_files) / 1024**2
print(f'dot_cache/  {len(cache_files):3d} files  {cache_mb:7.0f} MB')
print(f'TOTAL:      {pkg_mb + cache_mb:.0f} MB')

# Torch checkpoints
torch_ckpts = CACHE_OUT / 'torch' / 'hub' / 'checkpoints'
if torch_ckpts.exists():
    print('\n--- Torch hub checkpoints ---')
    for f in sorted(torch_ckpts.iterdir()):
        if f.is_file():
            print(f'  {f.name:50s} {f.stat().st_size/1024**2:7.1f} MB')

# HuggingFace models
hf_hub = CACHE_OUT / 'huggingface' / 'hub'
if hf_hub.exists():
    print('\n--- HuggingFace models ---')
    for d in sorted(hf_hub.iterdir()):
        if d.is_dir() and d.name.startswith('models--'):
            sz = sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / 1024**2
            print(f'  {d.name:50s} {sz:7.1f} MB')

# Checklist
print('\n--- Encoder Checklist ---')
expected = ['resnet34', 'resnet50', 'resnet101', 'efficientnet-b5',
            'convnext_base', 'convnext_large', 'mit_b3', 'mit_b5',
            'swinv2_base', 'swinv2_large']
all_files = ' '.join(f.name for f in CACHE_OUT.rglob('*') if f.is_file())
for enc in expected:
    found = enc.replace('-', '') in all_files.replace('-', '') or \
            enc.replace('_', '') in all_files.replace('_', '')
    icon = 'V' if found else 'X'
    print(f'  [{icon}] {enc}')

In [ ]:
print('=' * 60)
print('XONG! Lam theo cac buoc sau:')
print('=' * 60)
print()
print('BUOC 1: Save Version -> Quick Save')
print('        (doi 5-10 phut cho Kaggle save output)')
print()
print('BUOC 2: Mo notebook TRAIN')
print('        -> Internet OFF, GPU ON')
print()
print('BUOC 3: Add Data (2 nguon):')
print('        1. Tab "Notebook Output" -> tim NB01 nay -> Add')
print('           (chua packages + pretrained weights)')
print('        2. Dataset forest: uav-forest-sunny-part1 + uav-forest-sunny-next')
print('           (chua seq1-seq9 images)')
print()
print('BUOC 4: Chay notebook train')
print('        -> Cell dau tien se tu install packages + restore weights')